## hello-world 
### query

query 一次性的函数式 API, 没有记忆。

In [ ]:
from claude_agent_sdk import query
from dotenv import load_dotenv

load_dotenv(override=True)

from claude_agent_sdk import ClaudeAgentOptions

options = ClaudeAgentOptions(
    env={
        "ANTHROPIC_BASE_URL": "https://api.moonshot.cn/anthropic",
        "ANTHROPIC_MODEL": "kimi-k2.5",
        "ANTHROPIC_SMALL_FAST_MODEL": "kimi-k2.5",
        "ANTHROPIC_API_KEY": "sk-YO...",
    }
)

async for message in query(prompt="2+2=?", options=options):
    print(message)

SystemMessage(subtype='init', data={'type': 'system', 'subtype': 'init', 'cwd': '/home/anna/code/geek_note/agent_scaffold', 'session_id': 'a4db1741-6055-424c-a74b-0a37b4f1e2f9', 'tools': ['Task', 'AskUserQuestion', 'Bash', 'CronCreate', 'CronDelete', 'CronList', 'Edit', 'EnterPlanMode', 'EnterWorktree', 'ExitPlanMode', 'ExitWorktree', 'Glob', 'Grep', 'Monitor', 'NotebookEdit', 'PushNotification', 'Read', 'RemoteTrigger', 'ScheduleWakeup', 'Skill', 'TaskOutput', 'TaskStop', 'TodoWrite', 'WebFetch', 'WebSearch', 'Write'], 'mcp_servers': [], 'model': 'glm-4.7-flash', 'permissionMode': 'acceptEdits', 'slash_commands': ['update-config', 'debug', 'simplify', 'batch', 'fewer-permission-prompts', 'loop', 'schedule', 'claude-api', 'compact', 'context', 'cost', 'heapdump', 'init', 'review', 'security-review', 'insights', 'team-onboarding'], 'apiKeySource': 'ANTHROPIC_API_KEY', 'claude_code_version': '2.1.113', 'output_style': 'default', 'agents': ['Explore', 'general-purpose', 'Plan', 'statuslin

记录这里踩的坑：
1. kimi platform 和 kimi code，是2套 api(base url & key)，不能混用。一开始遇到了401,没留意到这个问题
2. 我本地使用了 cc-swith 管理claude code的api url和key管理。 .claude/settings.json 的优先级最高。我测试，.env 和 ClaudeAgentOptions 都失效。我fallback方案是 在 cc-switch 中使用1中配套的api url和key.
查了下说 SDK 在解析可执行文件路径时默认会优先找 PATH 里的 claude 命令。不了解里面的机制，后面留意下。

补充：用 ClaudeAgentOptions 中 参数 `setting_sources=[]`，表示 禁用所有文件系统设置，即可使用env参数中的配置。

In [ ]:
from claude_agent_sdk import query
from dotenv import load_dotenv
import os

load_dotenv(override=True)

from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions

options = ClaudeAgentOptions(
    env={
        "ANTHROPIC_BASE_URL": "https://api.moonshot.cn/anthropic",
        "ANTHROPIC_MODEL": "kimi-k2.5",
        "ANTHROPIC_SMALL_FAST_MODEL": "kimi-k2.5",
        "ANTHROPIC_API_KEY": os.getenv("ANTHROPIC_API_KEY"),
    },
    setting_sources=[],# 这个参数必须，否则还是会读 settings中的配置   
)

async for message in query(prompt="2+2=?", options=options):
    print(message)

SystemMessage(subtype='init', data={'type': 'system', 'subtype': 'init', 'cwd': '/home/anna/code/geek_note/agent_scaffold', 'session_id': 'ca6f57ae-24fb-447a-b2b7-86fb146b2ae4', 'tools': ['Task', 'AskUserQuestion', 'Bash', 'CronCreate', 'CronDelete', 'CronList', 'Edit', 'EnterPlanMode', 'EnterWorktree', 'ExitPlanMode', 'ExitWorktree', 'Glob', 'Grep', 'Monitor', 'NotebookEdit', 'PushNotification', 'Read', 'RemoteTrigger', 'ScheduleWakeup', 'Skill', 'TaskOutput', 'TaskStop', 'TodoWrite', 'WebFetch', 'WebSearch', 'Write'], 'mcp_servers': [], 'model': 'kimi-k2.5', 'permissionMode': 'default', 'slash_commands': ['update-config', 'debug', 'simplify', 'batch', 'fewer-permission-prompts', 'loop', 'schedule', 'claude-api', 'compact', 'context', 'cost', 'heapdump', 'init', 'review', 'security-review', 'insights', 'team-onboarding'], 'apiKeySource': 'ANTHROPIC_API_KEY', 'claude_code_version': '2.1.113', 'output_style': 'default', 'agents': ['Explore', 'general-purpose', 'Plan', 'statusline-setup'

### ClaudeAgentOptions

In [ ]:
options = ClaudeAgentOptions(
    system_prompt="你是一个有帮助的助手",  #系统提示词
    max_turns=3, #循环次数
    allowed_tools=["Read", "Write", "Bash"], #允许使用的工具，这几个都是内置工具
    permission_mode='acceptEdits', #设置工具权限，允许不经过人类确认，直接进行文件编辑
    cwd="/path/to/dir" #工作目录
)

Agent Loop: 一个循环：
```
①感知(输入/历史) → ②模型思考(LLM推理，决定下一步) → ③行动(调用工具) → ④观察(把结果喂回给模型) → 回到②
```
<!-- ![demo](./imgs/06_agent_loop.png) -->

<img src="./imgs/06_agent_loop.png" alt="agent loop demo" width="400">

直到模型判断任务已完成、达到最大轮次限制，或出错为止，才跳出循环。

和普通"一问一答"的最大区别是：模型自己决定要不要用工具、用哪个工具、用完之后要不要继续用，而不是由你的代码写死流程。

示例(claude官方：修bug)见目录 `./06_claudecode_sdk_agentloop`

### ClaudeSDKClient

In [3]:
async with ClaudeSDKClient() as client:
    await client.query("2+2=?")
    # Extract and print response
    async for msg in client.receive_response():
        print(msg)

SystemMessage(subtype='init', data={'type': 'system', 'subtype': 'init', 'cwd': '/home/anna/code/geek_note/agent_scaffold', 'session_id': 'd0b752b6-d614-461e-8c1e-c7cdcc764ecb', 'tools': ['Task', 'AskUserQuestion', 'Bash', 'CronCreate', 'CronDelete', 'CronList', 'Edit', 'EnterPlanMode', 'EnterWorktree', 'ExitPlanMode', 'ExitWorktree', 'Glob', 'Grep', 'Monitor', 'NotebookEdit', 'PushNotification', 'Read', 'RemoteTrigger', 'ScheduleWakeup', 'Skill', 'TaskOutput', 'TaskStop', 'TodoWrite', 'WebFetch', 'WebSearch', 'Write'], 'mcp_servers': [], 'model': 'glm-4.7-flash', 'permissionMode': 'acceptEdits', 'slash_commands': ['update-config', 'debug', 'simplify', 'batch', 'fewer-permission-prompts', 'loop', 'schedule', 'claude-api', 'compact', 'context', 'cost', 'heapdump', 'init', 'review', 'security-review', 'insights', 'team-onboarding'], 'apiKeySource': 'ANTHROPIC_API_KEY', 'claude_code_version': '2.1.113', 'output_style': 'default', 'agents': ['Explore', 'general-purpose', 'Plan', 'statuslin

## 生成金融研报

### 自定义工具抓取金融数据

使用免费的  [AKshare ](https://akshare.akfamily.xyz/data/stock/stock.html)，缺点网络不稳定(替代方案：Tushare,付费)。

避免抓取到的csv文件直接进入模型的上下文，使用按需加载。

In [1]:
from claude_agent_sdk import tool, create_sdk_mcp_server
import akshare as ak

@tool("getbalance", "获取沪深A股公司的资产负债表，并保存到文件中，其中参数stock_code是带市场标识的股票代码，比如SH600600，参数year是年份", {"stock_code": str, "year": str})
async def get_balance_sheet_A(stock_code: str = "SH600600", year: str = "2025"):
    try:  
        df_balance_sheet = ak.stock_balance_sheet_by_yearly_em(symbol="SH600600")
        # 只取REPORT_DATE是2025-12-31的数据
        df_balance_sheet = df_balance_sheet[df_balance_sheet['REPORT_DATE'] == f'{year}-12-31 00:00:00']
        # 获取项目根目录（假设当前文件在 0.1/tools/ 目录下）
        project_root = os.getcwd()
        # 去掉SH,SZ前缀
        #stock_code_clean = stock_code[2:] if stock_code.startswith(('SH', 'SZ')) else stock_code
        # 创建完整的文件路径
        filepath = os.path.join(project_root, "data", "financial_statements", f"{stock_code}_{year}_资产负债表.csv")
        # 创建目录（如果不存在）
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        
        # 使用指定目录保存文件
        df_balance_sheet.to_csv(filepath, index=False, encoding='utf-8-sig')
        return {
            "content": [
                {"type": "text", "text": f"资产负债表已保存到: {filepath}"}
            ]
        }
    except Exception as e:
            return {
                "content": [
                    {"type": "text", "text": f"获取资产负债表失败: {e}"}
                ]
            }

In [ ]:
# 将工具函数封装为一个本地的mcp server
server = create_sdk_mcp_server(
    name="financial-tools",
    version="1.0.0",
    tools=[get_balance_sheet_A]
)

# 将mcp server配置到options中
options = ClaudeAgentOptions(
    mcp_servers={"tools": server},
    # 格式：mcp__(固定的)__{mcp_server}(不固定的)__{tool_name}
    allowed_tools=["mcp__tools__getbalance"] # `mcp__` 是mcp tools名称默认前缀,'tools' 不固定，若上一行写‘abc’，则此处写`mcp__abc__getbalance`,2个地方一致即可
)

此处注意区分mcp server的用法。以前我单独启动一个进程，走 stdio or http 让客户端连接不一样，这里是 SDK内置mcp server,即 in-process/进程内模式。

- 没有额外进程，就在当前 Python 进程里
- 直接的 Python 函数调用（内存内传参）
- 性能开销几乎没有

Agent SDK 判断你传进去的是"配置字典"还是"server 对象"来决定走哪种模式：如果是像 {"command": "node", "args": [...]} 这样的配置，SDK 会帮你 spawn 一个子进程，用 stdio 通信；如果是像你这里直接传 create_sdk_mcp_server() 的返回值，SDK 就知道这是进程内 server，Claude 要调用 getbalance 这个工具时，直接在同一个进程里 await 你写的 get_balance_sheet_A 函数，不经过任何 subprocess 或网络。

In [5]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

options.env = {
        "ANTHROPIC_BASE_URL": "https://api.moonshot.cn/anthropic",
        "ANTHROPIC_MODEL": "kimi-k2.5",
        "ANTHROPIC_SMALL_FAST_MODEL": "kimi-k2.5",
        "ANTHROPIC_API_KEY": os.getenv("ANTHROPIC_API_KEY"),
    }
options.setting_sources = []

In [6]:
async with ClaudeSDKClient(options=options) as client:
    await client.query("获取 SH600600 的2025年度资产负债表")
    # Extract and print response
    async for msg in client.receive_response():
        print(msg)

SystemMessage(subtype='init', data={'type': 'system', 'subtype': 'init', 'cwd': '/home/anna/code/geek_note/agent_scaffold', 'session_id': '79108312-085c-40d8-8803-8ad65cf31782', 'tools': ['Task', 'AskUserQuestion', 'Bash', 'CronCreate', 'CronDelete', 'CronList', 'Edit', 'EnterPlanMode', 'EnterWorktree', 'ExitPlanMode', 'ExitWorktree', 'Glob', 'Grep', 'Monitor', 'NotebookEdit', 'PushNotification', 'Read', 'RemoteTrigger', 'ScheduleWakeup', 'Skill', 'TaskOutput', 'TaskStop', 'TodoWrite', 'WebFetch', 'WebSearch', 'Write', 'mcp__tools__getbalance'], 'mcp_servers': [{'name': 'tools', 'status': 'connected'}], 'model': 'kimi-k2.5', 'permissionMode': 'default', 'slash_commands': ['update-config', 'debug', 'simplify', 'batch', 'fewer-permission-prompts', 'loop', 'schedule', 'claude-api', 'compact', 'context', 'cost', 'heapdump', 'init', 'review', 'security-review', 'insights', 'team-onboarding'], 'apiKeySource': 'ANTHROPIC_API_KEY', 'claude_code_version': '2.1.113', 'output_style': 'default', '

  0%|          | 0/8 [00:00<?, ?it/s]

UserMessage(content=[ToolResultBlock(tool_use_id='mcp__tools__getbalance_0', content=[{'type': 'text', 'text': "资产负债表已保存到: /home/anna/code/geek_note/agent_scaffold/data/financial_statements/{'stock_code': 'SH600600', 'year': '2025'}_2025_资产负债表.csv"}], is_error=None)], uuid='7c021535-2973-4a61-b6f9-f33c8eabffe5', parent_tool_use_id=None, tool_use_result=[{'type': 'text', 'text': "资产负债表已保存到: /home/anna/code/geek_note/agent_scaffold/data/financial_statements/{'stock_code': 'SH600600', 'year': '2025'}_2025_资产负债表.csv"}])
AssistantMessage(content=[ThinkingBlock(thinking='工具返回了结果，显示资产负债表已保存到文件。让我读取这个文件的内容来展示给用户。', signature='')], model='kimi-k2.5', parent_tool_use_id=None, error=None, usage={'input_tokens': 18086, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 0, 'service_tier': 'standard', 'inference_geo': 'not_available', 'prompt_tokens': 18086, 'cached_tokens': 0}, message_id='chatcmpl-6a6868e61a9aa440369afc6e', stop_reason=None, session_id='79108312-085c-

此处第一次运行时缺少 ipywidgets 包，安装后，重启kernel

后记： 抓取数据的功能，不一定非要做成tool, 也可以定义为skills中1个script,只需在skill中定义好抓取数据的步骤，让LLM调用，既精确又节省上下文。

## 思考
### Skills 可以代替 LangGraph 等工作流吗?

TODO


### 什么样的业务代码适合封装为工具，什么样的业务代码适合作为 Skills 中的一个脚本？

像查询数据库数据，调用其他系统接口等，可以重复使用，作为底层的原子工具能力。
skill的脚本更偏向当前对当前执行流程中中间数据的处理，skill的专属工具。